# Moderation Data Analysis 

In [ ]:
import pandas as pd

In [17]:
df = pd.read_csv('content_moderation_1M.csv')
print('Data load successfuly')

Data load successfuly


### Data Quality Validation

In [19]:
df.head(5)

,ticket_id,content_id,moderator_id,moderator_name,team_lead_id,team_lead_name,shift_type,region,category_id,violation_category,...,report_source,content_type,created_timestamp,decision_timestamp,handling_time_seconds,action_taken,is_appealed,overturn_status,is_audited,qa_result
0,TICK-1504973,CONT-2504973,AI_AUTO,Automated System,TL_AI,Automated System,24/7,Global,CAT_01,Spam/Scam,...,User Flag,Video,2025-08-03 20:42:37.000000000,2025-08-03 20:42:51.813222519,0,Removed,False,NaN,False,NaN
1,TICK-1948058,CONT-2948058,MOD_268,Jeremy Reed,TL_23,Ethan Adams,Night,LATAM,CAT_02,Harassment/Bullying,...,User Flag,Image,2025-08-03 20:42:47.000000000,2025-08-03 20:55:32.303733952,48,Banned,False,NaN,False,NaN
2,TICK-1597884,CONT-2597884,AI_AUTO,Automated System,TL_AI,Automated System,24/7,Global,CAT_01,Spam/Scam,...,User Flag,Image,2025-08-03 20:43:17.000000000,2025-08-03 20:43:28.770195894,0,Removed,False,NaN,False,NaN
3,TICK-1904618,CONT-2904618,AI_AUTO,Automated System,TL_AI,Automated System,24/7,Global,CAT_01,Spam/Scam,...,User Flag,Text,2025-08-03 20:43:34.000000000,2025-08-03 20:43:57.635251705,0,Removed,False,NaN,False,NaN
4,TICK-1740076,CONT-2740076,AI_AUTO,Automated System,TL_AI,Automated System,24/7,Global,CAT_01,Spam/Scam,...,User Flag,Image,2025-08-03 20:43:56.000000000,2025-08-03 20:44:05.628533133,0,Kept,False,NaN,False,NaN


In [63]:
print('Raw data stats')
print('-'* 60)
print(df.columns)
print('-'* 60)
print(f'Rows , cols are :{df.shape}')
print('-'* 60)
print(df.info())
print('-'* 60)
print(df.describe())
print('-'* 60)
print(df.isnull().sum())
print('-'* 60)
print("Duplicate Ticket IDs :",df['ticket_id'].duplicated().sum())
print('-'* 60)
print("Duplicate content IDs :",df['content_id'].duplicated().sum())


Raw data stats
------------------------------------------------------------
Index(['ticket_id', 'content_id', 'moderator_id', 'moderator_name',
       'team_lead_id', 'team_lead_name', 'shift_type', 'region', 'category_id',
       'violation_category', 'severity_level', 'report_source', 'content_type',
       'created_timestamp', 'decision_timestamp', 'handling_time_seconds',
       'action_taken', 'is_appealed', 'overturn_status', 'is_audited',
       'qa_result'],
      dtype='object')
------------------------------------------------------------
Rows , cols are :(1000844, 21)
------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000844 entries, 0 to 1000843
Data columns (total 21 columns):
 #   Column                 Non-Null Count    Dtype 
---  ------                 --------------    ----- 
 0   ticket_id              1000844 non-null  object
 1   content_id             1000844 non-null  object
 2   moderator_id           1

### Findings

- Dataset contains **1,000,844 rows** and **21 columns**, suitable for large-scale analytics.
- `handling_time_seconds` is correctly stored as an integer, while timestamp columns need conversion to `datetime`.
- Missing values in `overturn_status` and `qa_result` appear to follow business rules and will be validated later.
- `region` has **169,160 missing values**, which requires further investigation.
- No duplicate `ticket_id` or `content_id` values were found, indicating unique records.
- Overall, the dataset is in good condition and ready for data quality validation and cleaning.

## Step 3: Data Cleaning

### Objective

Clean and prepare the dataset by resolving the issues identified during data profiling while preserving business logic and data integrity.

In [66]:
df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
df['decision_timestamp'] = pd.to_datetime(df['decision_timestamp'])

In [68]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000844 entries, 0 to 1000843
Data columns (total 21 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   ticket_id              1000844 non-null  object        
 1   content_id             1000844 non-null  object        
 2   moderator_id           1000844 non-null  object        
 3   moderator_name         1000844 non-null  object        
 4   team_lead_id           1000844 non-null  object        
 5   team_lead_name         1000844 non-null  object        
 6   shift_type             1000844 non-null  object        
 7   region                 831684 non-null   object        
 8   category_id            1000844 non-null  object        
 9   violation_category     1000844 non-null  object        
 10  severity_level         1000844 non-null  object        
 11  report_source          1000844 non-null  object        
 12  content_type           10008

### 3.2 Validate Missing Region Values

Investigate whether missing `region` values belong only to AI-generated records or indicate a data quality issue.

In [80]:
df.loc[df['region'].isna(),
       ['moderator_id','moderator_name','team_lead_id','shift_type']] \
  .drop_duplicates()

,moderator_id,moderator_name,team_lead_id,shift_type
8,MOD_312,Daniel Kane,TL_18,Evening
11,MOD_472,Austin Osborne,TL_25,Night
46,MOD_040,Jessica Callahan,TL_03,Morning
52,MOD_423,Shelly Spencer,TL_19,Evening
56,MOD_159,Ashley Barton,TL_13,Morning
...,...,...,...,...
1996,MOD_096,Larry Dixon,TL_09,Night
2127,MOD_178,Elizabeth Perkins,TL_25,Evening
2195,MOD_477,Brian Fitzgerald,TL_16,Morning
2739,MOD_182,Bryan Herrera,TL_25,Morning


In [86]:
# Check if moderator dimension has missing regions
moderators['region'].isna().sum()

np.int64(117)

In [88]:
df.loc[df['region'].isna(), 'moderator_id'].nunique()

117

### 3.2 Handle Missing Region Values

During data validation, missing `region` values were found for **117 moderators** in both the fact table and the moderator dimension. Since the source data does not provide the correct region, these missing values will be replaced with **"Unknown"** to maintain data consistency and improve reporting in Power BI. This preserves all records while making the missing information explicit.

In [95]:
df['region'] = df['region'].fillna('Unknown')
moderators['region'] = moderators['region'].fillna('Unknown')

### 3.3 Validate QA Results

Verify that `qa_result` is populated only for audited cases.

In [100]:
pd.crosstab(df['is_audited'],
            df['qa_result'].isna())

qa_result,False,True
is_audited,,
False,0,986411
True,14433,0


### 3.4 Validate Overturn Status

Verify that `overturn_status` is populated only for appealed cases.

In [103]:
pd.crosstab(df['is_appealed'],
            df['overturn_status'].isna())

overturn_status,False,True
is_appealed,,
False,0,920588
True,80256,0


### 3.5 Standardize Text Columns

Remove leading and trailing spaces from text columns.

In [106]:
df.columns = df.columns.str.strip()

## Data Cleaning Summary

- Converted timestamp columns to `datetime`.
- Standardized missing `region` values by replacing them with **"Unknown"** after verifying the issue originated from the source data.
- Validated that missing values in `qa_result` and `overturn_status` follow the expected business rules; no changes were made.
- Validated each cols name free of extra leading or trailing spaces
- The dataset is now cleaned, consistent, and ready for feature engineering and SQL analysis.

## Step 4: Feature Engineering

### Objective

Create additional date and time features from the timestamp columns to support trend analysis, time-based reporting, and interactive Power BI dashboards.

In [115]:
# Date Features

df['year'] = df['created_timestamp'].dt.year

df['quarter'] = 'Q' + df['created_timestamp'].dt.quarter.astype(str)

df['month'] = df['created_timestamp'].dt.month

df['month_name'] = df['created_timestamp'].dt.month_name()

df['day'] = df['created_timestamp'].dt.day

df['day_name'] = df['created_timestamp'].dt.day_name()

df['week'] = df['created_timestamp'].dt.isocalendar().week

df['hour'] = df['created_timestamp'].dt.hour

df['date'] = df['created_timestamp'].dt.date

In [139]:
df.head()

,ticket_id,content_id,moderator_id,moderator_name,team_lead_id,team_lead_name,shift_type,region,category_id,violation_category,...,qa_result,year,quarter,month,month_name,day,day_name,week,hour,date
0,TICK-1504973,CONT-2504973,AI_AUTO,Automated System,TL_AI,Automated System,24/7,Global,CAT_01,Spam/Scam,...,NaN,2025,Q3,8,August,3,Sunday,31,20,2025-08-03
1,TICK-1948058,CONT-2948058,MOD_268,Jeremy Reed,TL_23,Ethan Adams,Night,LATAM,CAT_02,Harassment/Bullying,...,NaN,2025,Q3,8,August,3,Sunday,31,20,2025-08-03
2,TICK-1597884,CONT-2597884,AI_AUTO,Automated System,TL_AI,Automated System,24/7,Global,CAT_01,Spam/Scam,...,NaN,2025,Q3,8,August,3,Sunday,31,20,2025-08-03
3,TICK-1904618,CONT-2904618,AI_AUTO,Automated System,TL_AI,Automated System,24/7,Global,CAT_01,Spam/Scam,...,NaN,2025,Q3,8,August,3,Sunday,31,20,2025-08-03
4,TICK-1740076,CONT-2740076,AI_AUTO,Automated System,TL_AI,Automated System,24/7,Global,CAT_01,Spam/Scam,...,NaN,2025,Q3,8,August,3,Sunday,31,20,2025-08-03


# Dim_tables Data profiling

In [9]:
moderators = pd.read_csv('dim_moderator.csv')
print('Data load successfuly')


Data load successfuly


In [142]:
moderators.head()

,moderator_id,moderator_name,team_lead_id,region,shift_type
0,MOD_001,Judy Baker,TL_20,Unknown,Morning
1,MOD_002,Justin Baker,TL_15,APAC,Night
2,MOD_003,Stephanie Ross,TL_11,EMEA,Night
3,MOD_004,Zachary Hicks,TL_08,EMEA,Night
4,MOD_005,Anthony Rodriguez,TL_21,APAC,Morning


In [154]:
print('Raw data stats')
print('-'* 60)
print(moderators.columns)
print('-'* 60)
print(f'Rows , cols are :{moderators.shape}')
print('-'* 60)
print(moderators.info())
print('-'* 60)
print(f'Null Values : {moderators.isnull().sum()}')
print('-'* 60)
print("Duplicate moderator_id IDs :",moderators['moderator_id'].duplicated().sum())
print('-'* 60)
print(moderators.nunique())

Raw data stats
------------------------------------------------------------
Index(['moderator_id', 'moderator_name', 'team_lead_id', 'region',
       'shift_type'],
      dtype='object')
------------------------------------------------------------
Rows , cols are :(501, 5)
------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501 entries, 0 to 500
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   moderator_id    501 non-null    object
 1   moderator_name  501 non-null    object
 2   team_lead_id    501 non-null    object
 3   region          501 non-null    object
 4   shift_type      501 non-null    object
dtypes: object(5)
memory usage: 19.7+ KB
None
------------------------------------------------------------
Null Values : moderator_id      0
moderator_name    0
team_lead_id      0
region            0
shift_type        0
dtype: int64
-------------------

### Findings

- The Moderator Dimension contains **501 records** and **5 columns**.
- No missing values were found in any column.
- `moderator_id` is unique for all records and can be used as the primary key.
- `moderator_name` contains **500 unique values**, indicating one moderator name is shared by two different moderator IDs. This will be verified to ensure the IDs remain unique.
- The dimension contains **26 Team Leads**, **5 Regions**, and **4 Shift Types**, which aligns with the expected business structure.

In [159]:
moderators[moderators.duplicated('moderator_name', keep=False)] \
.sort_values('moderator_name')

,moderator_id,moderator_name,team_lead_id,region,shift_type
199,MOD_200,Matthew Moore,TL_24,LATAM,Morning
250,MOD_251,Matthew Moore,TL_19,APAC,Night


### Duplicate Moderator Name Validation

A duplicate moderator name was identified; however, the corresponding `moderator_id` values are unique. Since `moderator_id` is the primary key, this is considered a valid scenario and does not require any data cleaning.

In [11]:
category = pd.read_csv('dim_category.csv')
print('Data load successfuly')

Data load successfuly


In [162]:
category.head()

,category_id,violation_category,severity_level
0,CAT_01,Spam/Scam,Low
1,CAT_02,Harassment/Bullying,Medium
2,CAT_03,Hate Speech,High
3,CAT_04,Sexual Content,High
4,CAT_05,Harm to Minors,Critical


In [166]:
print('Raw data stats')
print('-'* 60)
print(category.columns)
print('-'* 60)
print(f'Rows , cols are :{category.shape}')
print('-'* 60)
print(category.info())
print('-'* 60)
print(f'Null Values : {category.isnull().sum()}')
print('-'* 60)
print("Duplicate category_id IDs :",category['category_id'].duplicated().sum())
print('-'* 60)
print(category.nunique())

Raw data stats
------------------------------------------------------------
Index(['category_id', 'violation_category', 'severity_level'], dtype='object')
------------------------------------------------------------
Rows , cols are :(6, 3)
------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   category_id         6 non-null      object
 1   violation_category  6 non-null      object
 2   severity_level      6 non-null      object
dtypes: object(3)
memory usage: 276.0+ bytes
None
------------------------------------------------------------
Null Values : category_id           0
violation_category    0
severity_level        0
dtype: int64
------------------------------------------------------------
Duplicate category_id IDs : 0
----------------------------------------------------------

In [13]:
team_lead = pd.read_csv('dim_team_lead.csv')
print('Data load successfuly')


Data load successfuly


In [168]:
team_lead.head()

,team_lead_id,team_lead_name
0,TL_01,Allison Hill
1,TL_02,Noah Rhodes
2,TL_03,Angie Henderson
3,TL_04,Daniel Wagner
4,TL_05,Cristian Santos


In [170]:
print('Raw data stats')
print('-'* 60)
print(team_lead.columns)
print('-'* 60)
print(f'Rows , cols are :{team_lead.shape}')
print('-'* 60)
print(category.info())
print('-'* 60)
print(f'Null Values : {team_lead.isnull().sum()}')
print('-'* 60)
print("Duplicate team_lead_id IDs :",team_lead['team_lead_id'].duplicated().sum())
print('-'* 60)
print(team_lead.nunique())

Raw data stats
------------------------------------------------------------
Index(['team_lead_id', 'team_lead_name'], dtype='object')
------------------------------------------------------------
Rows , cols are :(26, 2)
------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   category_id         6 non-null      object
 1   violation_category  6 non-null      object
 2   severity_level      6 non-null      object
dtypes: object(3)
memory usage: 276.0+ bytes
None
------------------------------------------------------------
Null Values : team_lead_id      0
team_lead_name    0
dtype: int64
------------------------------------------------------------
Duplicate team_lead_id IDs : 0
------------------------------------------------------------
team_lead_id      26
team_lead_name    26
dtype:

## Step 5: Export Cleaned Data

### Objective

Export the cleaned fact and dimension tables as CSV files. These files will be imported into SQL Server for schema creation, business analysis, and Power BI dashboard development.

In [174]:

# Export cleaned datasets
df.to_csv("fact_content_moderation.csv", index=False)

moderators.to_csv("dim_moderator.csv", index=False)

team_lead.to_csv("dim_team_lead.csv", index=False)

category.to_csv("dim_category.csv", index=False)

print("✅ All cleaned files exported successfully.")

✅ All cleaned files exported successfully.


### ETL Summary

- Imported raw datasets.
- Performed data profiling and quality assessment.
- Cleaned and standardized the data.
- Engineered date-based features.
- Exported cleaned fact and dimension tables for SQL Server.

The datasets are now ready for data modeling and business analysis.